# Figure S1: Regional cumulative cases and deaths

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / "6_metapopulation_model" / "data").is_dir()),
    _here,
)
MODEL_DIR = REPO_ROOT / "6_metapopulation_model"
DATA_DIR = MODEL_DIR / "data"
FIGURES_DIR = MODEL_DIR / "visualization"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "DejaVu Sans"

timeline_events = pd.read_excel(
    DATA_DIR / "ppr_MA_2008_region_commune_agg.xlsx",
    sheet_name="Sheet1",
)
timeline_events["Start Date"] = pd.to_datetime(timeline_events["Start Date"])
timeline_events = timeline_events.dropna(subset=["REGION", "Start Date"])

INFECTED_REGIONS = (
    timeline_events.sort_values("Start Date")["REGION"]
    .drop_duplicates()
    .tolist()
)

In [ ]:
daily = (
    timeline_events
    .sort_values("Start Date")
    .groupby(["REGION", "Start Date"])
    .agg(outbreaks=("Start Date", "size"), cases=("Cases", "sum"), deaths=("Deaths", "sum"))
    .reset_index()
)
daily[["cum_outbreaks", "cum_cases", "cum_deaths"]] = (
    daily.groupby("REGION")[["outbreaks", "cases", "deaths"]].cumsum()
)

fig, axes = plt.subplots(2, 1, figsize=(7.2, 6.2), sharex=True, dpi=300)
metrics = [
    ("cum_cases",     "Cumulative cases"),
    ("cum_deaths",    "Cumulative deaths"),
]
panel_labels = ["A", "B"]

REG_COLORS = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Olive
]

for ax, (metric, ylabel), label in zip(axes, metrics, panel_labels):
    for i, region in enumerate(INFECTED_REGIONS):
        grp = daily[daily["REGION"] == region]
        if not grp.empty:
            ax.plot(
                grp["Start Date"], grp[metric], label=region,
                linewidth=1.8, alpha=0.85, color=REG_COLORS[i]
            )

    ax.set_ylabel(ylabel, fontweight="bold")
    ax.grid(False)
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.set_title(label, loc="left", fontsize=12, fontweight="bold")

axes[-1].set_xlabel("Epidemiological Week (2008)")
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[1].legend(loc="upper left", ncol=1, fontsize=8, frameon=False)
fig.autofmt_xdate(rotation=45)
fig.tight_layout()

base = FIGURES_DIR / "Figure_S1_Region_Cumulative_All"
fig.savefig(
    f"{base}.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.08,
)
plt.show()
print(f"Saved: {base}.png")